In [1]:
pip install --upgrade huggingface_hub

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 30.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 134.8 MB/s  0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [huggingface_hub] [huggingface_hub]
Note: you may need to restart the kernel to use updated packages.


In [2]:
from huggingface_hub import login
login()

In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
splits = {'train': 'train.csv', 'test': 'test.csv'}
df = pd.read_csv("hf://datasets/adilbekovich/Sentiment140Twitter/" + splits["train"])

In [4]:
df.head()

,text,label
0,Closet organizer install complete. Now for th...,0
1,Mornin' All!! ....I need to wake up....this w...,1
2,@Lega_c ahhhhhhhhhh! he suxxxxxxx! he claimed ...,0
3,@endlessblush Haha. I guess all the good bits ...,0
4,family guy funny,1


In [5]:
# Definición de los archivos según tu notebook
splits = {'train': 'train.csv', 'test': 'test.csv'}
base_url = "hf://datasets/adilbekovich/Sentiment140Twitter/"

# Bucle para descargar y guardar cada uno
for name, filename in splits.items():
    print(f"Descargando {name}...")
    # Carga el dataframe desde Hugging Face
    temp_df = pd.read_csv(base_url + filename)
    
    # Lo guarda en tu carpeta actual con el mismo nombre
    temp_df.to_csv(filename, index=False)
    print(f"Archivo '{filename}' guardado correctamente.")

Descargando train...
Archivo 'train.csv' guardado correctamente.
Descargando test...
Archivo 'test.csv' guardado correctamente.


In [6]:
def csv_to_parquet_chunked(input_file, output_folder, chunksize=100_000):
    
    # Crear carpeta si no existe
    os.makedirs(output_folder, exist_ok=True)
    
    chunk_number = 0
    
    for chunk in pd.read_csv(input_file, chunksize=chunksize):
        
        output_path = os.path.join(
            output_folder,
            f"part_{chunk_number}.parquet"
        )
        
        chunk.to_parquet(
            output_path,
            engine="pyarrow",
            index=False
        )
        
        print(f"Chunk {chunk_number} guardado en {output_path}")
        chunk_number += 1

    print("Conversión completada")


# Ejecutar
csv_to_parquet_chunked("train.csv", "train_parquet")
csv_to_parquet_chunked("test.csv", "test_parquet")

Chunk 0 guardado en train_parquet/part_0.parquet
Chunk 1 guardado en train_parquet/part_1.parquet
Chunk 2 guardado en train_parquet/part_2.parquet
Chunk 3 guardado en train_parquet/part_3.parquet
Chunk 4 guardado en train_parquet/part_4.parquet
Chunk 5 guardado en train_parquet/part_5.parquet
Chunk 6 guardado en train_parquet/part_6.parquet
Chunk 7 guardado en train_parquet/part_7.parquet
Chunk 8 guardado en train_parquet/part_8.parquet
Chunk 9 guardado en train_parquet/part_9.parquet
Chunk 10 guardado en train_parquet/part_10.parquet
Chunk 11 guardado en train_parquet/part_11.parquet
Chunk 12 guardado en train_parquet/part_12.parquet
Chunk 13 guardado en train_parquet/part_13.parquet
Conversión completada
Chunk 0 guardado en test_parquet/part_0.parquet
Chunk 1 guardado en test_parquet/part_1.parquet
Chunk 2 guardado en test_parquet/part_2.parquet
Conversión completada


In [ ]:
import s3fs

load_dotenv()

# Inicializar el sistema de archivos de S3
fs = s3fs.S3FileSystem()

# Definir el bucket destino (Capa RAW - Bronce)
S3_RAW_PATH = f"{os.getenv('S3_BUCKET_NAME')}/data/raw"

print(f"Iniciando subida a s3://{S3_RAW_PATH}...")

# 1. Subir la carpeta completa de train
print("Subiendo particiones de Entrenamiento (train_parquet)...")
fs.put("train_parquet", f"{S3_RAW_PATH}/train_parquet", recursive=True)
print("train_parquet subido exitosamente.")

# 2. Subir la carpeta completa de test
print("Subiendo particiones de Prueba (test_parquet)...")
fs.put("test_parquet", f"{S3_RAW_PATH}/test_parquet", recursive=True)
print("test_parquet subido exitosamente.")

print("¡Fase 1 (Ingesta a Capa RAW) completada al 100%!")

Iniciando subida a s3://parcial-pln/data/raw...
Subiendo particiones de Entrenamiento (train_parquet)...
train_parquet subido exitosamente.
Subiendo particiones de Prueba (test_parquet)...
test_parquet subido exitosamente.
¡Fase 1 (Ingesta a Capa RAW) completada al 100%!
